In [1]:
# ============================================================
# CELL 1: Install required packages
# ============================================================

!pip install -q -U "transformers>=4.55.0" accelerate kernels

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 87.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 92.1 MB/s eta 0:00:00:00:01


In [2]:
# ============================================================
# CELL 2: Imports and configuration
# ============================================================

import os
import gc
import re
import time
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

import matplotlib.pyplot as plt
import seaborn as sns


# -----------------------------
# Reproducibility
# -----------------------------

RANDOM_SEED = 42

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# -----------------------------
# Experiment configuration
# -----------------------------

MODEL_ID = "openai/gpt-oss-20b"

TEST_SIZE = 1000

REASONING_EFFORT = "low"

# IMPORTANT:
# 8 was too short and GPT-OSS stopped in the analysis channel.
MAX_NEW_TOKENS = 64

# We are doing one example at a time because
# GPT-OSS-20B nearly fills the T4's VRAM.
BATCH_SIZE = 1

SHOW_EVERY_INSTANCE = False
SHOW_PROGRESS_EVERY = 50

DATA_PATH = "/kaggle/input/datasets/amermahbub01/sentifive"

TEXT_COLUMN = "data"
GOLD_COLUMN = "title"


print("Configuration loaded.")
print("Model:", MODEL_ID)
print("Samples:", TEST_SIZE)
print("Reasoning effort:", REASONING_EFFORT)
print("Max new tokens:", MAX_NEW_TOKENS)

Configuration loaded.
Model: openai/gpt-oss-20b
Samples: 1000
Reasoning effort: low
Max new tokens: 64


In [3]:
# ============================================================
# CELL 3: GPU check
# ============================================================

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)

if torch.cuda.is_available():

    gpu_name = torch.cuda.get_device_name(0)

    total_memory = (
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3
    )

    print("GPU:", gpu_name)
    print(f"GPU memory: {total_memory:.2f} GB")

else:
    raise RuntimeError(
        "CUDA GPU was not detected. "
        "Please enable GPU in Kaggle."
    )

PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
GPU memory: 14.56 GB


In [4]:
# ============================================================
# CELL 4: Load SentiFive dataset
# ============================================================

import os
import glob

print("Dataset directory:")
print(DATA_PATH)

print("\nFiles found:")

for root, dirs, files in os.walk(DATA_PATH):
    for file in files:
        print(os.path.join(root, file))

Dataset directory:
/kaggle/input/datasets/amermahbub01/sentifive

Files found:
/kaggle/input/datasets/amermahbub01/sentifive/SentiFive.csv


In [5]:
# ============================================================
# CELL 4B: Locate CSV
# ============================================================

csv_files = glob.glob(
    os.path.join(DATA_PATH, "**", "*.csv"),
    recursive=True
)

print("CSV files found:", len(csv_files))

for file in csv_files:
    print(file)

if len(csv_files) == 0:
    raise FileNotFoundError(
        "No CSV file was found inside the SentiFive dataset directory."
    )

CSV_PATH = csv_files[0]

print("\nUsing:")
print(CSV_PATH)

CSV files found: 1
/kaggle/input/datasets/amermahbub01/sentifive/SentiFive.csv

Using:
/kaggle/input/datasets/amermahbub01/sentifive/SentiFive.csv


In [6]:
# ============================================================
# CELL 4C: Read dataset
# ============================================================

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (31411, 2)

Columns:
['data', 'title']

First 5 rows:


,data,title
0,থিওরি অফ রিলেটিভিটির প্রেক্ষিতে বাংলা ভাষার এ...,নিশ্চিত ইতিবাচক
1,এডমিনকে অনুরোধ করব যারা এসব কাজ করে দুই বাংলার...,নিরপেক্ষ
2,খাদ্যগলিডেকার্স লেনে ভোজনচর্চা,নিরপেক্ষ
3,আদনান তোর বৌয়ের ভোদা খাবো খুব মিষ্টি রস বেরোয...,নিশ্চিত নেতিবাচক
4,কে কে আমার ভিডিও লাইক করলে,নিরপেক্ষ


In [7]:
# ============================================================
# CELL 5: Dataset verification
# ============================================================

if TEXT_COLUMN not in df.columns:
    raise ValueError(
        f"Text column '{TEXT_COLUMN}' was not found."
    )

if GOLD_COLUMN not in df.columns:
    raise ValueError(
        f"Gold label column '{GOLD_COLUMN}' was not found."
    )

print("Text column:", TEXT_COLUMN)
print("Gold label column:", GOLD_COLUMN)

print("\nDataset size:", len(df))

print("\nMissing values:")
print(df[[TEXT_COLUMN, GOLD_COLUMN]].isna().sum())

print("\nGold label distribution:")
print(df[GOLD_COLUMN].value_counts())

Text column: data
Gold label column: title

Dataset size: 31411

Missing values:
data     0
title    0
dtype: int64

Gold label distribution:
title
নিশ্চিত নেতিবাচক    6780
নিশ্চিত ইতিবাচক     6537
নিরপেক্ষ            6453
কিছুটা নেতিবাচক     6006
কিছুটা ইতিবাচক      5635
Name: count, dtype: int64


In [8]:
# ============================================================
# CELL 6: Define sentiment labels
# ============================================================

LABELS = [
    "নিশ্চিত নেতিবাচক",
    "সামান্য নেতিবাচক",
    "নিরপেক্ষ",
    "সামান্য ইতিবাচক",
    "নিশ্চিত ইতিবাচক"
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

ID_TO_LABEL = {
    i: label
    for i, label in enumerate(LABELS)
}

print("Label mapping:")

for i, label in ID_TO_LABEL.items():
    print(i, "->", label)

Label mapping:
0 -> নিশ্চিত নেতিবাচক
1 -> সামান্য নেতিবাচক
2 -> নিরপেক্ষ
3 -> সামান্য ইতিবাচক
4 -> নিশ্চিত ইতিবাচক


In [9]:
# ============================================================
# SELECT 3000 SAMPLES
# ============================================================

from sklearn.model_selection import train_test_split

if TEST_SIZE > len(df):

    raise ValueError(
        f"TEST_SIZE={TEST_SIZE} is larger than "
        f"dataset size={len(df)}"
    )

if TEST_SIZE == len(df):

    experiment_df = df.copy()

else:

    experiment_df, _ = train_test_split(
        df,
        train_size=TEST_SIZE,
        stratify=df[GOLD_COLUMN],
        random_state=RANDOM_SEED
    )

    experiment_df = experiment_df.copy()

# Reset index
experiment_df = experiment_df.reset_index(drop=True)

print("=" * 80)
print("EXPERIMENT DATA")
print("=" * 80)

print(
    "Samples selected:",
    len(experiment_df)
)

print("\nGold distribution:")

distribution = (
    experiment_df[GOLD_COLUMN]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

display(distribution)

EXPERIMENT DATA
Samples selected: 1000

Gold distribution:


title
নিশ্চিত নেতিবাচক    216
সামান্য নেতিবাচক      0
নিরপেক্ষ            206
সামান্য ইতিবাচক       0
নিশ্চিত ইতিবাচক     208
Name: count, dtype: int64

In [10]:
# ============================================================
# CELL 7: Stratified 3,000-sample evaluation set
# ============================================================

from sklearn.model_selection import train_test_split

if TEST_SIZE > len(df):
    raise ValueError(
        f"TEST_SIZE={TEST_SIZE} is larger than dataset size={len(df)}"
    )

experiment_df, _ = train_test_split(
    df,
    test_size=len(df) - TEST_SIZE,
    stratify=df[GOLD_COLUMN],
    random_state=RANDOM_SEED
)

experiment_df = experiment_df.reset_index(drop=True)

print("Evaluation sample size:", len(experiment_df))

print("\nSample label distribution:")
print(experiment_df[GOLD_COLUMN].value_counts())

print("\nSample percentage:")
print(
    (experiment_df[GOLD_COLUMN].value_counts(normalize=True) * 100)
    .round(2)
)

Evaluation sample size: 1000

Sample label distribution:
title
নিশ্চিত নেতিবাচক    216
নিশ্চিত ইতিবাচক     208
নিরপেক্ষ            206
কিছুটা নেতিবাচক     191
কিছুটা ইতিবাচক      179
Name: count, dtype: int64

Sample percentage:
title
নিশ্চিত নেতিবাচক    21.6
নিশ্চিত ইতিবাচক     20.8
নিরপেক্ষ            20.6
কিছুটা নেতিবাচক     19.1
কিছুটা ইতিবাচক      17.9
Name: proportion, dtype: float64


In [11]:
# ============================================================
# CELL 8: Load GPT-OSS-20B
# ============================================================

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID
)

print("Tokenizer loaded.")

print("\nLoading GPT-OSS-20B...")
print("This may take several minutes.")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto"
)

model.eval()

print("\nGPT-OSS-20B loaded successfully.")

print("Model device:", model.device)

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]

...

metadata.json: 0.00B [00:00, ?B/s]

Fetching ... files: 0it [00:00, ?it/s]

Loading weights:   0%|          | 0/411 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/177 [00:00<?, ?B/s]


GPT-OSS-20B loaded successfully.
Model device: cuda:0


In [12]:
# ============================================================
# CELL 9: GPU memory check
# ============================================================

if torch.cuda.is_available():

    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved = torch.cuda.memory_reserved() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print(f"GPU total:     {total:.2f} GB")
    print(f"GPU allocated: {allocated:.2f} GB")
    print(f"GPU reserved:  {reserved:.2f} GB")
    print(f"GPU free:      {total - allocated:.2f} GB")

GPU total:     14.56 GB
GPU allocated: 5.53 GB
GPU reserved:  5.55 GB
GPU free:      9.04 GB


In [13]:
# ============================================================
# CELL 10: Classification prompt
# ============================================================

SYSTEM_PROMPT = """
You are a Bengali sentiment classification model.

Classify the sentiment of the given Bengali text into exactly ONE of these classes:

0 = নিশ্চিত নেতিবাচক
1 = সামান্য নেতিবাচক
2 = নিরপেক্ষ
3 = সামান্য ইতিবাচক
4 = নিশ্চিত ইতিবাচক

Return ONLY the single digit:
0, 1, 2, 3, or 4.

Do not return explanations.
Do not return any other text.
""".strip()


def create_messages(text):
    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": str(text)
        }
    ]

In [14]:
# ============================================================
# CELL 11: GPT-OSS output extraction
# ============================================================

def extract_final_channel(text):
    """
    Extract the final response from GPT-OSS Harmony-style output.
    """

    if text is None:
        return ""

    text = str(text)

    final_marker = "<|channel|>final<|message|>"

    if final_marker in text:

        final_part = text.split(
            final_marker,
            1
        )[1]

        if "<|end|>" in final_part:
            final_part = final_part.split(
                "<|end|>",
                1
            )[0]

        return final_part.strip()

    return text.strip()

In [15]:
# ============================================================
# CELL 12: Parse model prediction
# ============================================================

def parse_prediction(text):
    """
    Extract a valid class ID 0-4 from GPT-OSS output.
    """

    if text is None:
        return None

    text = str(text).strip()

    # First look for a clean standalone digit.
    match = re.search(
        r"(?<!\d)([0-4])(?!\d)",
        text
    )

    if match:
        return int(match.group(1))

    # Handle accidental Bengali digit output.
    bengali_to_english = str.maketrans(
        "০১২৩৪",
        "01234"
    )

    converted = text.translate(
        bengali_to_english
    )

    match = re.search(
        r"(?<!\d)([0-4])(?!\d)",
        converted
    )

    if match:
        return int(match.group(1))

    # Handle Bengali/English label names if model ignores
    # the numeric-only instruction.

    text_lower = text.lower()

    english_mapping = {
        "strongly negative": 0,
        "slightly negative": 1,
        "neutral": 2,
        "slightly positive": 3,
        "strongly positive": 4,
    }

    for label, idx in english_mapping.items():

        if label in text_lower:
            return idx

    bengali_mapping = {
        "নিশ্চিত নেতিবাচক": 0,
        "সামান্য নেতিবাচক": 1,
        "নিরপেক্ষ": 2,
        "সামান্য ইতিবাচক": 3,
        "নিশ্চিত ইতিবাচক": 4,
    }

    for label, idx in bengali_mapping.items():

        if label in text:
            return idx

    return None

In [16]:
# ============================================================
# CELL 13: Single GPT-OSS prediction
# ============================================================

def predict_one(text):

    messages = create_messages(text)

    # -----------------------------------------
    # Apply GPT-OSS chat template
    # -----------------------------------------

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        reasoning_effort=REASONING_EFFORT
    )

    # Move tensors to the model's device.
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    input_length = inputs["input_ids"].shape[-1]

    # -----------------------------------------
    # Generate
    # -----------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    # -----------------------------------------
    # Remove prompt tokens
    # -----------------------------------------

    generated_tokens = outputs[
        0,
        input_length:
    ]

    raw_output = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=False
    ).strip()

    # -----------------------------------------
    # Extract final channel
    # -----------------------------------------

    final_output = extract_final_channel(
        raw_output
    )

    # -----------------------------------------
    # Parse class
    # -----------------------------------------

    prediction = parse_prediction(
        final_output
    )

    return {
        "raw_output": raw_output,
        "final_output": final_output,
        "prediction_id": prediction
    }

In [17]:
# ============================================================
# CELL 14: Test one example
# ============================================================

test_text = experiment_df.iloc[0][TEXT_COLUMN]
test_gold = experiment_df.iloc[0][GOLD_COLUMN]

print("TEST TEXT")
print(test_text)

print("\nGOLD LABEL")
print(test_gold)

print("\nGOLD ID")
print(LABEL_TO_ID.get(test_gold))

print("\nRunning GPT-OSS...")

test_result = predict_one(
    test_text
)

print("\nRAW OUTPUT")
print(repr(test_result["raw_output"]))

print("\nFINAL OUTPUT")
print(repr(test_result["final_output"]))

print("\nPREDICTION ID")
print(test_result["prediction_id"])

if test_result["prediction_id"] is not None:

    print("\nPREDICTION LABEL")
    print(
        ID_TO_LABEL[
            test_result["prediction_id"]
        ]
    )
else:
    print("\nWARNING: Prediction could not be parsed.")

TEST TEXT
বেচারা পাইসা গেছে বেয়ানছোত কা বাচ্চা

GOLD LABEL
নিশ্চিত নেতিবাচক

GOLD ID
0

Running GPT-OSS...

RAW OUTPUT
'<|channel|>analysis<|message|>Sentiment? The text: "বেচারা পাইসা গেছে বেয়ানছোত কা বাচ্চা" seems negative, complaining. So class 0.<|end|><|start|>assistant<|channel|>final<|message|>0<|return|>'

FINAL OUTPUT
'0<|return|>'

PREDICTION ID
0

PREDICTION LABEL
নিশ্চিত নেতিবাচক


In [18]:
# ============================================================
# CELL 15: Run GPT-OSS on all 1,000 samples
# ============================================================

results = []

total_samples = len(experiment_df)

start_time = time.time()

print("=" * 70)
print("STARTING GPT-OSS-20B EVALUATION")
print("=" * 70)

print(f"Total samples: {total_samples}")
print(f"Reasoning effort: {REASONING_EFFORT}")
print(f"Max new tokens: {MAX_NEW_TOKENS}")
print()

for idx, row in experiment_df.iterrows():

    text = row[TEXT_COLUMN]
    gold_label = row[GOLD_COLUMN]

    gold_id = LABEL_TO_ID.get(
        gold_label
    )

    try:

        result = predict_one(
            text
        )

        prediction_id = result[
            "prediction_id"
        ]

        prediction_label = (
            ID_TO_LABEL.get(prediction_id)
            if prediction_id is not None
            else None
        )

        results.append({
            "index": idx,
            "text": text,
            "gold_label": gold_label,
            "gold_id": gold_id,
            "raw_output": result["raw_output"],
            "final_output": result["final_output"],
            "prediction_id": prediction_id,
            "prediction_label": prediction_label
        })

    except Exception as e:

        print(
            f"\nERROR at sample {idx + 1}: {e}"
        )

        results.append({
            "index": idx,
            "text": text,
            "gold_label": gold_label,
            "gold_id": gold_id,
            "raw_output": None,
            "final_output": None,
            "prediction_id": None,
            "prediction_label": None
        })

    # -----------------------------------------
    # Progress reporting
    # -----------------------------------------

    completed = idx + 1

    if (
        completed % SHOW_PROGRESS_EVERY == 0
        or completed == total_samples
    ):

        elapsed = time.time() - start_time

        avg_time = elapsed / completed

        remaining = (
            avg_time *
            (total_samples - completed)
        )

        print(
            f"[{completed}/{total_samples}] "
            f"Elapsed: {elapsed/60:.1f} min | "
            f"Avg: {avg_time:.2f} sec/sample | "
            f"Estimated remaining: {remaining/60:.1f} min"
        )

    # -----------------------------------------
    # Optional per-instance display
    # -----------------------------------------

    if SHOW_EVERY_INSTANCE:

        print("\n" + "-" * 60)

        print("TEXT:")
        print(text)

        print("\nGOLD:")
        print(gold_label)

        print("\nMODEL FINAL:")
        print(result["final_output"])

        print("\nPREDICTION:")
        print(prediction_label)

        print("-" * 60)

    # -----------------------------------------
    # Do NOT call empty_cache every iteration.
    # -----------------------------------------

STARTING GPT-OSS-20B EVALUATION
Total samples: 1000
Reasoning effort: low
Max new tokens: 64

[50/1000] Elapsed: 9.3 min | Avg: 11.17 sec/sample | Estimated remaining: 176.9 min
[100/1000] Elapsed: 15.7 min | Avg: 9.41 sec/sample | Estimated remaining: 141.1 min
[150/1000] Elapsed: 21.4 min | Avg: 8.58 sec/sample | Estimated remaining: 121.5 min
[200/1000] Elapsed: 27.8 min | Avg: 8.33 sec/sample | Estimated remaining: 111.1 min
[250/1000] Elapsed: 34.4 min | Avg: 8.26 sec/sample | Estimated remaining: 103.2 min
[300/1000] Elapsed: 40.1 min | Avg: 8.03 sec/sample | Estimated remaining: 93.6 min
[350/1000] Elapsed: 46.2 min | Avg: 7.93 sec/sample | Estimated remaining: 85.9 min
[400/1000] Elapsed: 52.1 min | Avg: 7.82 sec/sample | Estimated remaining: 78.2 min
[450/1000] Elapsed: 58.2 min | Avg: 7.76 sec/sample | Estimated remaining: 71.1 min
[500/1000] Elapsed: 64.4 min | Avg: 7.73 sec/sample | Estimated remaining: 64.4 min
[550/1000] Elapsed: 70.4 min | Avg: 7.68 sec/sample | Estimate

In [19]:
# ============================================================
# CELL 16: Results DataFrame
# ============================================================

results_df = pd.DataFrame(results)

print("Finished!")

print("\nResults shape:")
print(results_df.shape)

display(
    results_df.head(10)
)

Finished!

Results shape:
(1000, 8)


,index,text,gold_label,gold_id,raw_output,final_output,prediction_id,prediction_label
0,0,বেচারা পাইসা গেছে বেয়ানছোত কা বাচ্চা,নিশ্চিত নেতিবাচক,0.0,<|channel|>analysis<|message|>Sentiment? The t...,0<|return|>,0.0,নিশ্চিত নেতিবাচক
1,1,ভিডিও খুব ভালো ছিলো আর ভাবিকে সুন্দর লেগেছে ভা...,কিছুটা ইতিবাচক,NaN,<|channel|>analysis<|message|>Sentiment positi...,4<|return|>,4.0,নিশ্চিত ইতিবাচক
2,2,পুলিশের সাথে একটা কুকুর ও দেখতে পেলাম বাহহ এরা...,কিছুটা নেতিবাচক,NaN,<|channel|>analysis<|message|>The text is nons...,2<|return|>,2.0,নিরপেক্ষ
3,3,আপনি প্রয়োজন ছাড়াই হাসেন,নিশ্চিত ইতিবাচক,4.0,"<|channel|>analysis<|message|>Sentiment: ""আপনি...",0<|return|>,0.0,নিশ্চিত নেতিবাচক
4,4,তর চেহারা দেখলেই বোঝা য়ায় তর মোখে রোচি নাই,কিছুটা ইতিবাচক,NaN,<|channel|>analysis<|message|>Sentiment: negat...,0<|return|>,0.0,নিশ্চিত নেতিবাচক
5,5,গাজাখোর জাতির গাজাখোর পুলিশ চুদি বাংলাদেশেরে,নিশ্চিত নেতিবাচক,0.0,<|channel|>analysis<|message|>The user text is...,0<|return|>,0.0,নিশ্চিত নেতিবাচক
6,6,ওদের কে কুওা চুদে জন্ম দিছে,নিশ্চিত নেতিবাচক,0.0,<|channel|>analysis<|message|>The user text is...,0<|return|>,0.0,নিশ্চিত নেতিবাচক
7,7,শত্রুর ও যদি কোন প্রশংসার দিক থাকে সেই প্রশংসা...,নিশ্চিত ইতিবাচক,4.0,<|channel|>analysis<|message|>Sentiment: somew...,3<|return|>,3.0,সামান্য ইতিবাচক
8,8,পুলিশ ভাই আপনাকে স্যালুট জানাই,কিছুটা নেতিবাচক,NaN,"<|channel|>analysis<|message|>Text: ""পুলিশ ভাই...",4<|return|>,4.0,নিশ্চিত ইতিবাচক
9,9,জাজ গুলো সব বাজে ছেলেদের কে অপমান কিরছে,কিছুটা ইতিবাচক,NaN,<|channel|>analysis<|message|>Sentiment: negat...,0<|return|>,0.0,নিশ্চিত নেতিবাচক


In [20]:
# ============================================================
# CELL 17: Prediction parsing diagnostics
# ============================================================

total = len(results_df)

valid_predictions = (
    results_df["prediction_id"]
    .notna()
    .sum()
)

invalid_predictions = (
    results_df["prediction_id"]
    .isna()
    .sum()
)

print("Total samples:", total)
print("Valid predictions:", valid_predictions)
print("Invalid predictions:", invalid_predictions)

print(
    f"\nParsing success rate: "
    f"{valid_predictions / total * 100:.2f}%"
)

print("\nPrediction distribution:")

print(
    results_df[
        "prediction_label"
    ].value_counts(dropna=False)
)

Total samples: 1000
Valid predictions: 945
Invalid predictions: 55

Parsing success rate: 94.50%

Prediction distribution:
prediction_label
নিশ্চিত নেতিবাচক    407
নিরপেক্ষ            291
নিশ্চিত ইতিবাচক     190
None                 55
সামান্য নেতিবাচক     36
সামান্য ইতিবাচক      21
Name: count, dtype: int64


In [21]:
# ============================================================
# CELL 18: Inspect invalid predictions
# ============================================================

invalid_df = results_df[
    results_df["prediction_id"].isna()
]

print(
    "Number of invalid predictions:",
    len(invalid_df)
)

if len(invalid_df) > 0:

    display(
        invalid_df[
            [
                "text",
                "gold_label",
                "raw_output",
                "final_output"
            ]
        ].head(20)
    )
else:

    print(
        "Excellent: every prediction was successfully parsed."
    )

Number of invalid predictions: 55


,text,gold_label,raw_output,final_output
64,নুরু ভাইকে যে যাই বলুক না কেন নুরু ভাই শত মার ...,নিশ্চিত ইতিবাচক,<|channel|>analysis<|message|>We need sentimen...,<|channel|>analysis<|message|>We need sentimen...
78,বিঃদ্রঃ আমিও মাদ্রাসায় পড়িনাই সিএসইর ছাত্র ছিল...,কিছুটা ইতিবাচক,<|channel|>analysis<|message|>Sentiment? The t...,<|channel|>analysis<|message|>Sentiment? The t...
107,ওসি চাকরের মুখে হাসি আর কি সরলতা দেখলে হাসি পা...,কিছুটা নেতিবাচক,<|channel|>analysis<|message|>Sentiment: negat...,<|channel|>analysis<|message|>Sentiment: negat...
108,এই দেশে এরকম হওয়াটা অবাক করার মতো নতুন কোন বিষ...,নিশ্চিত ইতিবাচক,<|channel|>analysis<|message|>Sentiment: somew...,<|channel|>analysis<|message|>Sentiment: somew...
123,দাদা ভুল করলেন কফি হাউজ মানে মাটন কবিরাজি,নিশ্চিত ইতিবাচক,<|channel|>analysis<|message|>Sentiment? The u...,
125,হিসাব করেন যখন একটা লোক কোটি টাকা খরচ করে সামা...,নিরপেক্ষ,<|channel|>analysis<|message|>We need sentimen...,<|channel|>analysis<|message|>We need sentimen...
132,অভিযান তদন্ত পুলিশ ষ্টেশনে করেন আসল অপরাধী ইউন...,কিছুটা নেতিবাচক,<|channel|>analysis<|message|>We need sentimen...,<|channel|>analysis<|message|>We need sentimen...
135,আপনারা মিডিয়া তাদের আর্তনাত কি বোঝবেন ও বলতে চ...,নিশ্চিত ইতিবাচক,<|channel|>analysis<|message|>We need sentimen...,<|channel|>analysis<|message|>We need sentimen...
264,আরে ভাই এখান থেকে তো রাজনীতি করে বড় হচ্ছে আর য...,নিশ্চিত নেতিবাচক,<|channel|>analysis<|message|>We need sentimen...,<|channel|>analysis<|message|>We need sentimen...
266,ধন্য সেই ক্যামেরাম্যান যে চোখের সামনে আপনার খা...,নিশ্চিত ইতিবাচক,<|channel|>analysis<|message|>Sentiment: somew...,<|channel|>analysis<|message|>Sentiment: somew...


In [24]:
print("Missing gold labels:")
print(results_df[results_df["gold_id"].isna()][["text", "gold_label"]].head(20))

print("\nNumber of missing gold labels:")
print(results_df["gold_id"].isna().sum())

Missing gold labels:
                                                 text       gold_label
1   ভিডিও খুব ভালো ছিলো আর ভাবিকে সুন্দর লেগেছে ভা...   কিছুটা ইতিবাচক
2   পুলিশের সাথে একটা কুকুর ও দেখতে পেলাম বাহহ এরা...  কিছুটা নেতিবাচক
4          তর চেহারা দেখলেই বোঝা য়ায় তর মোখে রোচি নাই   কিছুটা ইতিবাচক
8                      পুলিশ ভাই আপনাকে স্যালুট জানাই  কিছুটা নেতিবাচক
9             জাজ গুলো সব বাজে ছেলেদের কে অপমান কিরছে   কিছুটা ইতিবাচক
12       কি শুনলাম আমি ওনার বউ ওনাকে ভাইয়া বলে ডাকছেন  কিছুটা নেতিবাচক
15  ওদেরকে শান্ত করে লাভ নাই ওরা শান্ত জাতি না  এ ...  কিছুটা নেতিবাচক
21  আমার ও মেদ বেড়ে গেছে ধন্যবাদ এমন একটা খবর এর জন্য   কিছুটা ইতিবাচক
22  গতকাল রাত থেকে ঘটনা ঘটছে আমি নিজে ও ঝামেলায় প...  কিছুটা নেতিবাচক
28  আমি তো আগেই চিন্তা করতাম দৈনিক এতো মরা মুরগী য...  কিছুটা নেতিবাচক
29                 বালা পালানির ক্রিমও বেজাল বাংলাদেশ  কিছুটা নেতিবাচক
33                    ইউটিউবের এবাউটে দেয়া আছে ভাইয়া    কিছুটা ইতিবাচক
35                    ভাইয়া নাপা শুনে বেশি মজা পাইলাম   

In [25]:
# ============================================================
# CELL 19: Prepare evaluation data
# ============================================================

evaluation_df = results_df[
    results_df["prediction_id"].notna() &
    results_df["gold_id"].notna()
].copy()

evaluation_df["prediction_id"] = (
    evaluation_df["prediction_id"]
    .astype(int)
)

evaluation_df["gold_id"] = (
    evaluation_df["gold_id"]
    .astype(int)
)

print("Total samples:", len(results_df))
print("Valid evaluation samples:", len(evaluation_df))

print(
    "Excluded invalid predictions:",
    results_df["prediction_id"].isna().sum()
)

print(
    "Excluded invalid gold labels:",
    results_df["gold_id"].isna().sum()
)

Total samples: 1000
Valid evaluation samples: 599
Excluded invalid predictions: 55
Excluded invalid gold labels: 370


In [26]:
# ============================================================
# CELL 20: Accuracy
# ============================================================

y_true = evaluation_df["gold_id"].values
y_pred = evaluation_df["prediction_id"].values

accuracy = accuracy_score(
    y_true,
    y_pred
)

print(
    f"Accuracy: {accuracy:.4f}"
)

print(
    f"Accuracy (%): {accuracy * 100:.2f}%"
)

Accuracy: 0.6027
Accuracy (%): 60.27%


In [28]:
# ============================================================
# CELL 21: Overall metrics
# ============================================================

macro_precision, macro_recall, macro_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(5)),
        average="macro",
        zero_division=0
    )
)

weighted_precision, weighted_recall, weighted_f1, _ = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(5)),
        average="weighted",
        zero_division=0
    )
)

print("=" * 50)
print("OVERALL METRICS")
print("=" * 50)

print(
    f"Accuracy          : {accuracy:.4f}"
)

print(
    f"Macro Precision   : {macro_precision:.4f}"
)

print(
    f"Macro Recall      : {macro_recall:.4f}"
)

print(
    f"Macro F1          : {macro_f1:.4f}"
)

print(
    f"Weighted F1       : {weighted_f1:.4f}"
)

OVERALL METRICS
Accuracy          : 0.6027
Macro Precision   : 0.3841
Macro Recall      : 0.3608
Macro F1          : 0.3677
Weighted F1       : 0.6133


In [29]:
# ============================================================
# CELL 22: Class-wise metrics
# ============================================================

precision, recall, f1, support = (
    precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(5)),
        average=None,
        zero_division=0
    )
)

class_metrics = pd.DataFrame({
    "Class ID": range(5),
    "Class": LABELS,
    "Precision": precision,
    "Recall": recall,
    "F1": f1,
    "Support": support
})

print("=" * 70)
print("CLASS-WISE METRICS")
print("=" * 70)

display(
    class_metrics
)

CLASS-WISE METRICS


,Class ID,Class,Precision,Recall,F1,Support
0,0,নিশ্চিত নেতিবাচক,0.635983,0.748768,0.687783,203
1,1,সামান্য নেতিবাচক,0.000000,0.000000,0.000000,0
2,2,নিরপেক্ষ,0.569231,0.555000,0.562025,200
3,3,সামান্য ইতিবাচক,0.000000,0.000000,0.000000,0
4,4,নিশ্চিত ইতিবাচক,0.715328,0.500000,0.588589,196


In [30]:
# ============================================================
# CELL 23: Classification report
# ============================================================

print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(5)),
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)

                  precision    recall  f1-score   support

নিশ্চিত নেতিবাচক     0.6360    0.7488    0.6878       203
সামান্য নেতিবাচক     0.0000    0.0000    0.0000         0
        নিরপেক্ষ     0.5692    0.5550    0.5620       200
 সামান্য ইতিবাচক     0.0000    0.0000    0.0000         0
 নিশ্চিত ইতিবাচক     0.7153    0.5000    0.5886       196

        accuracy                         0.6027       599
       macro avg     0.3841    0.3608    0.3677       599
    weighted avg     0.6397    0.6027    0.6133       599



In [31]:
# ============================================================
# CELL 24: Confusion Matrix
# ============================================================

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(5))
)

cm_df = pd.DataFrame(
    cm,
    index=LABELS,
    columns=LABELS
)

print("Confusion Matrix:")
display(cm_df)

Confusion Matrix:


,নিশ্চিত নেতিবাচক,সামান্য নেতিবাচক,নিরপেক্ষ,সামান্য ইতিবাচক,নিশ্চিত ইতিবাচক
নিশ্চিত নেতিবাচক,152,2,37,3,9
সামান্য নেতিবাচক,0,0,0,0,0
নিরপেক্ষ,47,9,111,3,30
সামান্য ইতিবাচক,0,0,0,0,0
নিশ্চিত ইতিবাচক,40,5,47,6,98


In [33]:
# ============================================================
# CELL 25: Label distribution
# ============================================================

gold_counts = (
    results_df["gold_label"]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

pred_counts = (
    evaluation_df["prediction_label"]
    .value_counts()
    .reindex(LABELS, fill_value=0)
)

label_distribution = pd.DataFrame({
    "Label": LABELS,
    "Gold Count": gold_counts.values,
    "GPT Predicted Count": pred_counts.values
})

print("=" * 70)
print("LABEL DISTRIBUTION")
print("=" * 70)

display(
    label_distribution
)

LABEL DISTRIBUTION


,Label,Gold Count,GPT Predicted Count
0,নিশ্চিত নেতিবাচক,216,239
1,সামান্য নেতিবাচক,0,16
2,নিরপেক্ষ,206,195
3,সামান্য ইতিবাচক,0,12
4,নিশ্চিত ইতিবাচক,208,137


In [35]:
# ============================================================
# CELL 27: Final summary
# ============================================================

elapsed_total = time.time() - start_time

summary = pd.DataFrame({
    "Metric": [
        "Model",
        "Total Samples",
        "Valid Predictions",
        "Invalid Predictions",
        "Parsing Success",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Total Runtime (minutes)",
        "Average Time / Sample (seconds)"
    ],

    "Value": [
        MODEL_ID,
        len(results_df),
        len(evaluation_df),
        len(results_df) - len(evaluation_df),
        f"{len(evaluation_df) / len(results_df) * 100:.2f}%",
        f"{accuracy:.4f}",
        f"{macro_precision:.4f}",
        f"{macro_recall:.4f}",
        f"{macro_f1:.4f}",
        f"{weighted_f1:.4f}",
        f"{elapsed_total / 60:.2f}",
        f"{elapsed_total / len(results_df):.2f}"
    ]
})

display(summary)

,Metric,Value
0,Model,openai/gpt-oss-20b
1,Total Samples,1000
2,Valid Predictions,599
3,Invalid Predictions,401
4,Parsing Success,59.90%
5,Accuracy,0.6027
6,Macro Precision,0.3841
7,Macro Recall,0.3608
8,Macro F1,0.3677
9,Weighted F1,0.6133
